# Data Extraction and Anonymization with RTC-NER-Extended model

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import spacy
spacy.require_cpu()
from spacy.tokens import DocBin
from tqdm import tqdm
import json

In [ ]:
!python -m spacy info


============================== Info about spaCy ==============================

spaCy version    3.8.14                        
Location         /usr/local/lib/python3.12/dist-packages/spacy
Platform         Linux-6.6.122+-x86_64-with-glibc2.35
Python version   3.12.13                       
Pipelines        en_core_web_sm (3.8.0)        



In [ ]:
data = pd.read_csv('rtc_final.csv', encoding='utf-8')
rtc_df = data[['rtc_id', 'news_date', 'content', 'road', 'landmark', 'suburb', 'town', 'lga',
       'state', 'hospital', 'rtc_site', 'rtc_lat', 'rtc_long', 'casualty', 'injured']]
rtc_df.head()

,rtc_id,news_date,content,road,landmark,suburb,town,lga,state,hospital,rtc_site,rtc_lat,rtc_long,casualty,injured
0,0,6/17/2024,three persons have died in an accident that oc...,NaN,NaN,NaN,aagba,NaN,osun,NaN,"aagba, osun state",7.898776,4.713527,3,0
1,1,3/10/2024,twenty-one members of united evangelical churc...,aba-owerri road,united evangelical church abayi international,NaN,aba,NaN,abia,abia state university teaching hospital aba,"united evangelical church abayi international,...",5.120455,7.342642,0,21
2,2,7/17/2018,five persons identified as members of the laza...,enugu-onitsha expressway,NaN,NaN,abakaliki,NaN,ebonyi,NaN,abakaliki,6.323061,8.112012,0,5
3,3,5/4/2021,four persons have been confirmed dead with fo...,ogoja-abakaliki-enugu highway,NaN,NaN,abakaliki,NaN,ebonyi,alex ekwueme federal teaching hospital,"spera n deo junction, abakaliki",6.309542,8.102372,4,4
4,4,11/14/2016,the federal road safety corps (frsc) has confi...,NaN,hill top-water works junction,NaN,abakaliki,NaN,ebonyi,NaN,"hill top-water works junction, abakaliki",6.325140,8.114399,1,0


In [ ]:
rtc_df.columns

Index(['rtc_id', 'news_date', 'content', 'road', 'landmark', 'suburb', 'town',
       'lga', 'state', 'hospital', 'rtc_site', 'rtc_lat', 'rtc_long',
       'casualty', 'injured'],
      dtype='object')

### Use the `RTC_NER_EXT` model to extract all contextual entities from each news

In [ ]:
rtc_df.shape

(794, 15)

In [ ]:
#Load the newly creaed model
rtc_ner_ext = spacy.load('rtc_ner_ext_model/model-best')

In [ ]:
ner = rtc_ner_ext.get_pipe("ner")

print(sorted(ner.labels))

['ADDRESS', 'FACTORS', 'NO_VEHICLES', 'PERSON', 'PERSONS_INVOLVED', 'PLATE_NO', 'TIME', 'VEHICLE_TYPE', 'VICTIM_ORGANIZATION', 'WEEKDAY']


In [ ]:
from collections import defaultdict

def extract_entities(text):
    """
    Extracts all entities from text and groups them by label.
    Returns comma-separated strings for each entity type.
    """
    entities = defaultdict(list)
    doc = rtc_ner_ext(text)

    for ent in doc.ents:
        #entities[ent.label_].append(ent.text)
        if ent.text not in entities[ent.label_]:
          entities[ent.label_].append(ent.text)

    # Convert lists to comma-separated strings
    entities = {k: ", ".join(v) for k, v in entities.items()}

    return entities


# Create empty lists to store extracted entities
time = []
weekday = []
vehicle_type = []
no_vehicles = []
persons_involved = []
factors = []
person = []
plate_no = []
victim_org = []


# Iterate through each article in the DataFrame
for content in rtc_df["content"]:
  # Extract entities using the function
  article_entities = extract_entities(content)

  # Append entities to respective lists
  time.append(article_entities.get("TIME", ""))
  weekday.append(article_entities.get("WEEKDAY", ""))
  vehicle_type.append(article_entities.get("VEHICLE_TYPE", ""))
  no_vehicles.append(article_entities.get("NO_VEHICLES", ""))
  persons_involved.append(article_entities.get("PERSONS_INVOLVED", ""))
  factors.append(article_entities.get("FACTORS", ""))
  person.append(article_entities.get("PERSON", ""))
  plate_no.append(article_entities.get("PLATE_NO", ""))
  victim_org.append(article_entities.get("VICTIM_ORGANIZATION", ""))
  victim_address.append(article_entities.get("'ADDRESS'", ""))


# Add the extracted entities to the dataframe
rtc_df = rtc_df.assign(
    time = time,
    weekday = weekday,
    vehicle_type = vehicle_type,
    no_vehicles = no_vehicles,
    persons_involved = persons_involved,
    factors = factors,
    person = person,
    plate_no = plate_no,
    victim_org = victim_org,
    victim_address = victim_address)

In [ ]:
rtc_df.head(10)

,rtc_id,news_date,content,road,landmark,suburb,town,lga,state,hospital,...,injured,time,weekday,vehicle_type,no_vehicles,persons_involved,factors,person,plate_no,victim_org
0,0,6/17/2024,three persons have died in an accident that oc...,NaN,NaN,NaN,aagba,NaN,osun,NaN,...,0,,sunday.,"motorcycle, toyota camry",,three passengers,,adeleke kehinde,akd 810 qx.,
1,1,3/10/2024,twenty-one members of united evangelical churc...,aba-owerri road,united evangelical church abayi international,NaN,aba,NaN,abia,abia state university teaching hospital aba,...,21,6 pm,thursday,lexus sports utility vehicle,,twenty-one,"pregnant woman, hit by the vehicle from behind...",nnamdi kelvin,,"united evangelical church, abayi international"
2,2,7/17/2018,five persons identified as members of the laza...,enugu-onitsha expressway,NaN,NaN,abakaliki,NaN,ebonyi,NaN,...,5,,three-day,mercedes 190 car,,five persons,"muoka-led, skidded off the road, brake system....","kalu, mr. sunday ajayi",mus 984 bx,federal road safety corp (frsc)
3,3,5/4/2021,four persons have been confirmed dead with fo...,ogoja-abakaliki-enugu highway,NaN,NaN,abakaliki,NaN,ebonyi,alex ekwueme federal teaching hospital,...,4,,monday,articulated vehicle,,,"junction, females, males, child, male","mrs stella uchegbu, uchegbu",,
4,4,11/14/2016,the federal road safety corps (frsc) has confi...,NaN,hill top-water works junction,NaN,abakaliki,NaN,ebonyi,NaN,...,0,,sunday,"articulated vehicle, motorcycle.",,,"junction., motorcyclist, reckless driving, spe...","mr sunday iyamah, iyamah",,
5,5,1/8/2023,a woman has been killed in an accident at the ...,abeokuta-ibadan expressway,mobil petrol station,obantoko,abeokuta,NaN,ogun,NaN,...,2,9.10am,sunday.,motorcycle,,three pillion,"motorcyclist, hit her head, fell on her side, ...","babatunde akinbiyi, akinbiyi",,
6,6,12/28/2019,the federal road safety corps (frsc) said 13 p...,abeokuta-sagamu expressway,muhammadu buhari estate,NaN,abeokuta,NaN,ogun,NaN,...,13,,friday,mazda bus,,18 persons,"excessive driving, violation of traffic rules....",florence okpe,aaa921xt.,
7,7,4/16/2023,there was tension on the abeokuta-sagamu route...,abeokuta-sagamu expressway,NaN,okemosan,abeokuta,NaN,ogun,federal medical centre abeokuta,...,0,5 am,saturday.,,,,"motorcyclist, night, unknown vehicle, speed","florence okpe, okpe, ahmed umar, umar",,
8,8,5/15/2018,two persons were on monday night confirmed dea...,adigbe-obada road,NaN,NaN,abeokuta,NaN,ogun,hope hospital adigbe abeokuta,...,4,7.30 pm,monday,"nissan pickup van, volvo truck, nissan pick-up...",,six people,"night, recklessness, ran into, stationed, stat...","mr babatunde akinbiyi, akinbiyi","fff 147 aa., mus 285 xq.",
9,9,4/28/2024,the ogun state command of federal road safety ...,lagos-abeokuta expressway,NaN,NaN,abeokuta,NaN,ogun,the general hospital,...,6,7:10 am,,"sino truck, howo truck, bajaj tricycle",,seven persons,"mechanical deficiency, excessive speeding, hit...","anthony uga, uga","fst 926 ya, gsw 180 xa, kle 876 kt.",


In [ ]:
rtc_df.shape

(794, 24)

In [ ]:
# convert all values to lowercase
cols = [
    'time', 'weekday', 'vehicle_type', 'no_vehicles',
    'persons_involved', 'factors'
]

rtc_df[cols] = rtc_df[cols].apply(
    lambda col: col.str.lower() if col.dtype == 'object' else col
)

In [ ]:
# You can clean those columns by removing all special characters except hyphen (-) like this:

import re

cols = [
    'weekday', 'vehicle_type', 'no_vehicles',
    'persons_involved', 'factors'
]

pattern = r"[^a-zA-Z0-9\s\-]"

for col in cols:
    rtc_df[col] = rtc_df[col].apply(
        lambda x: re.sub(r"\s+", " ", re.sub(pattern, '', x)).strip()
        if isinstance(x, str) else x
    )

In [ ]:
rtc_df.columns

Index(['rtc_id', 'news_date', 'content', 'road', 'landmark', 'suburb', 'town',
       'lga', 'state', 'hospital', 'rtc_site', 'rtc_lat', 'rtc_long',
       'casualty', 'injured', 'time', 'weekday', 'vehicle_type', 'no_vehicles',
       'persons_involved', 'factors', 'person', 'plate_no', 'victim_org'],
      dtype='object')

#### Anonimyze the content column by replacing the entities identified by our  NER model with the respective label names 'PERSON','PLATE_NO', 'ADDRESS' and 'VICTIM_ORGANIZATION'

In [ ]:
def anonymize_text(text):
    """
    Replace NER entities in text with their label names.
    Only anonymizes PERSON, PLATE_NO, ADDRESS, VICTIM_ORGANIZATION.
    """
    doc = rtc_ner_ext(text)

    # Only anonymize selected labels
    target_labels = {"PERSON", "PLATE_NO", "ADDRESS", "VICTIM_ORGANIZATION"}

    # Build replacements from end to start (important to preserve indices)
    spans = [
        (ent.start_char, ent.end_char, ent.label_)
        for ent in doc.ents
        if ent.label_ in target_labels
    ]

    # Sort in reverse to avoid shifting text positions
    spans = sorted(spans, key=lambda x: x[0], reverse=True)

    anonymized_text = text

    for start, end, label in spans:
        anonymized_text = (
            anonymized_text[:start] + label + anonymized_text[end:]
        )

    return anonymized_text

rtc_df['content'] = rtc_df['content'].apply(anonymize_text)

In [ ]:
rtc_df['content'].head()

,content
0,three persons have died in an accident that oc...
1,"twenty-one members of VICTIM_ORGANIZATION, in ..."
2,five persons identified as members of the laza...
3,four persons have been confirmed dead with fo...
4,the federal road safety corps (frsc) has confi...
